<a href="https://colab.research.google.com/github/SamarBabar02/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamarBabar02/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### Rule

I will prioritize pages that have low CTR but relatively strong search visibility. These pages are already appearing in search results, but they receive relatively few clicks. This makes them potential opportunities for improving the page's search result appeal and content relevance.

For the baseline score, lower CTR will increase the priority score, while better search position will also increase the priority score.

### Reason Codes

- `LOW_CTR_STRONG_POSITION` — Low CTR with a strong search position (1–3).
- `LOW_CTR_MODERATE_POSITION` — Low CTR with a moderate search position (4–10).
- `LOW_CTR_WEAK_POSITION` — Low CTR with a weak search position (>10).

### Action Label

- `REVIEW_CTR` — Page should be reviewed for a possible CTR/content improvement opportunity.

In [1]:
!pip -q install duckdb huggingface_hub

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face secret configured successfully.")

Hugging Face secret configured successfully.


In [3]:
rel = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
"""

In [4]:
query = f"""
SELECT *
FROM {rel}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
"""

df = con.sql(query).df()

print("Shape:", df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (9841378, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [5]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [6]:
print("Start date:", df["report_date"].min())
print("End date:", df["report_date"].max())
print("Total rows:", len(df))

Start date: 2026-03-01 00:00:00
End date: 2026-03-31 00:00:00
Total rows: 9841378


In [7]:
df[['gsc_impressions',
    'gsc_clicks',
    'gsc_avg_position',
    'ga4_pageviews',
    'ga4_sessions',
    'ga4_engaged_sessions',
    'ga4_total_engagement_sec',
    'scroll_events']].describe()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_engaged_sessions,ga4_total_engagement_sec,scroll_events
count,9.841378e+06,9.841378e+06,3.611061e+06,6822637.0,6822637.0,6822637.0,6822637.0,6822637.0
mean,2.851812e+01,8.350782e-02,1.582665e+01,0.217637,0.190514,0.004331,0.698905,0.032261
std,1.559266e+02,7.814341e-01,1.985603e+01,2.142851,1.96875,0.076647,17.206612,0.413166
min,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0
25%,0.000000e+00,0.000000e+00,3.742120e+00,0.0,0.0,0.0,0.0,0.0
50%,0.000000e+00,0.000000e+00,7.500000e+00,0.0,0.0,0.0,0.0,0.0
75%,6.000000e+00,0.000000e+00,2.020000e+01,0.0,0.0,0.0,0.0,0.0
max,4.008400e+04,2.740000e+02,4.980000e+02,875.0,792.0,21.0,7083.0,254.0


In [8]:
df[['gsc_impressions',
    'gsc_clicks',
    'gsc_avg_position',
    'ga4_pageviews',
    'ga4_sessions',
    'ga4_engaged_sessions',
    'ga4_total_engagement_sec',
    'scroll_events']].isnull().mean().sort_values(ascending=False)

,0
gsc_avg_position,0.633074
ga4_pageviews,0.306740
ga4_engaged_sessions,0.306740
ga4_sessions,0.306740
ga4_total_engagement_sec,0.306740
scroll_events,0.306740
gsc_impressions,0.000000
gsc_clicks,0.000000


In [11]:
# Calculate CTR only where impressions are greater than 0
import numpy as np
import pandas as pd
df['ctr'] = np.where(
    df['gsc_impressions'] > 0,
    df['gsc_clicks'] / df['gsc_impressions'],
    np.nan
)

# Bucket CTR
df['ctr_bucket'] = pd.cut(
    df['ctr'],
    bins=[-np.inf, 0.02, 0.05, np.inf],
    labels=['Low', 'Medium', 'High']
)

# Bucket average position
df['position_bucket'] = pd.cut(
    df['gsc_avg_position'],
    bins=[-np.inf, 3, 10, np.inf],
    labels=['Strong (1-3)', 'Moderate (4-10)', 'Weak (>10)']
)

# CTR bucket table
ctr_table = (
    df.groupby('ctr_bucket', observed=False)
      .size()
      .reset_index(name='n')
)

print("CTR Bucket Table")
display(ctr_table)

# Position bucket table
position_table = (
    df.groupby('position_bucket', observed=False)
      .size()
      .reset_index(name='n')
)

print("\nPosition Bucket Table")
display(position_table)

CTR Bucket Table


,ctr_bucket,n
0,Low,3510885
1,Medium,65464
2,High,34712



Position Bucket Table


,position_bucket,n
0,Strong (1-3),727362
1,Moderate (4-10),1456122
2,Weak (>10),1427577


In [12]:
# Check CTR vs position together

ctr_position_table = pd.crosstab(
    df['position_bucket'],
    df['ctr_bucket'],
    margins=True
)

display(ctr_position_table)

ctr_bucket,Low,Medium,High,All
position_bucket,,,,
Strong (1-3),703452,14960,8950,727362
Moderate (4-10),1405902,33983,16237,1456122
Weak (>10),1401531,16521,9525,1427577
All,3510885,65464,34712,3611061


### My baseline rule

I will prioritize content pages that have relatively strong search positions but low CTR. The idea is to identify pages that already appear in stronger search positions but receive relatively few clicks compared with their impressions. This is a directional baseline for finding possible refresh or CTR-improvement opportunities, not proof that the page needs a refresh.

### Signals checked

1. **CTR** — calculated as clicks divided by impressions.
   - Verdict: **CONFIRMED**
   - The data contains a large number of low-CTR observations, including pages in strong positions (1–3). This supports using low CTR as a directional signal for the baseline.

2. **Average position**
   - Verdict: **CONFIRMED**
   - The data contains substantial observations across strong (1–3), moderate (4–10), and weak (>10) position buckets, making position useful for separating ranking strength.

### Reason codes

The rule will output one reason code:

- **LOW_CTR_STRONG_POSITION** — the page has a strong search position but low CTR.

### Action label

The action label will be:

- **REVIEW_CTR** — review the page for a possible CTR/refresh opportunity.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.